In [1]:
# Create a script that specifically focuses on pairing the main MPRAGE and PET_FDG scans, 
# ignoring the preprocessed segments (swc files) and other scan types.

import os
import pandas as pd
import nibabel as nib
import numpy as np
from pathlib import Path
from tqdm import tqdm

class MRIPETOrganizer:
    def __init__(self, base_path):
        self.base_path = Path(base_path)
        
    def find_pairs(self):
        """Find matching MPRAGE (MRI) and PET_FDG pairs"""
        print("Finding MRI-PET pairs...")
        pairs = []
        all_files = list(self.base_path.glob("*.nii"))
        
        # Group files by subject ID and timepoint
        subject_files = {}
        for file_path in tqdm(all_files):
            filename = file_path.name
            
            # Skip preprocessed files (swc) and other scan types
            if 'swc' in filename or 'TSE' in filename:
                continue
                
            # Extract subject ID (e.g., '003_S_1057')
            parts = filename.split('_')
            if len(parts) < 4:
                continue
                
            subject_id = '_'.join(parts[:3])
            timepoint = parts[3]  # e.g., '2008-02'
            
            if subject_id not in subject_files:
                subject_files[subject_id] = {}
            if timepoint not in subject_files[subject_id]:
                subject_files[subject_id][timepoint] = {'mri': None, 'pet': None}
            
            # Categorize as MRI or PET
            if 'MPRAGE.nii' in filename and 'swc' not in filename:
                subject_files[subject_id][timepoint]['mri'] = file_path
            elif 'PET_FDG' in filename:
                subject_files[subject_id][timepoint]['pet'] = file_path
        
        # Create pairs only when both MRI and PET exist
        valid_pairs = []
        for subject_id, timepoints in subject_files.items():
            for timepoint, files in timepoints.items():
                if files['mri'] and files['pet']:
                    valid_pairs.append({
                        'subject_id': subject_id,
                        'timepoint': timepoint,
                        'mri_path': str(files['mri']),
                        'pet_path': str(files['pet'])
                    })
        
        print(f"\nFound {len(valid_pairs)} valid MRI-PET pairs")
        return valid_pairs
    
    def verify_pairs(self, pairs):
        """Verify that all pairs can be loaded and have expected dimensions"""
        print("\nVerifying pairs can be loaded correctly...")
        verified_pairs = []
        
        for pair in tqdm(pairs):
            try:
                # Load MRI
                mri_img = nib.load(pair['mri_path'])
                mri_data = mri_img.get_fdata()
                
                # Load PET
                pet_img = nib.load(pair['pet_path'])
                pet_data = pet_img.get_fdata()
                
                # Verify dimensions match
                if mri_data.shape == pet_data.shape:
                    pair['dimensions'] = mri_data.shape
                    pair['mri_value_range'] = (float(np.min(mri_data)), float(np.max(mri_data)))
                    pair['pet_value_range'] = (float(np.min(pet_data)), float(np.max(pet_data)))
                    verified_pairs.append(pair)
                
            except Exception as e:
                print(f"\nError processing pair for subject {pair['subject_id']}: {str(e)}")
                
        print(f"\nVerified {len(verified_pairs)} pairs out of {len(pairs)}")
        return verified_pairs
    
    def save_pairs_info(self, pairs, output_file='mri_pet_pairs.csv'):
        """Save pair information to CSV file"""
        df = pd.DataFrame(pairs)
        df.to_csv(output_file, index=False)
        print(f"\nPair information saved to {output_file}")
        
    def print_summary(self, pairs):
        """Print summary statistics of the pairs"""
        if not pairs:
            print("No valid pairs found!")
            return
            
        subjects = len(set(pair['subject_id'] for pair in pairs))
        timepoints = len(set(pair['timepoint'] for pair in pairs))
        
        print("\n=== Dataset Summary ===")
        print(f"Total valid pairs: {len(pairs)}")
        print(f"Unique subjects: {subjects}")
        print(f"Unique timepoints: {timepoints}")
        print(f"Average pairs per subject: {len(pairs)/subjects:.2f}")
        
        # Print example pair
        print("\nExample pair structure:")
        print(pd.DataFrame([pairs[0]]).to_string())

def main():
    # Initialize organizer
    adni_path = r"E:/KHU Gangdong Hospital Data/PET1_FDG/ADNI1"
    organizer = MRIPETOrganizer(adni_path)
    
    # Find and verify pairs
    pairs = organizer.find_pairs()
    verified_pairs = organizer.verify_pairs(pairs)
    
    # Save and summarize results
    organizer.save_pairs_info(verified_pairs)
    organizer.print_summary(verified_pairs)
    
    return verified_pairs

if __name__ == "__main__":
    pairs = main()

Finding MRI-PET pairs...


100%|██████████| 5777/5777 [00:00<00:00, 381402.40it/s]



Found 627 valid MRI-PET pairs

Verifying pairs can be loaded correctly...


100%|██████████| 627/627 [00:17<00:00, 35.90it/s] 



Verified 627 pairs out of 627

Pair information saved to mri_pet_pairs.csv

=== Dataset Summary ===
Total valid pairs: 627
Unique subjects: 305
Unique timepoints: 37
Average pairs per subject: 2.06

Example pair structure:
   subject_id timepoint                                                                                 mri_path                                                                                        pet_path    dimensions mri_value_range                           pet_value_range
0  003_S_1057   2008-02  E:\KHU Gangdong Hospital Data\PET1_FDG\ADNI1\003_S_1057_2008-02_sw003_S_1057_MPRAGE.nii  E:\KHU Gangdong Hospital Data\PET1_FDG\ADNI1\003_S_1057_2008-02_wr003_S_1057_PET_FDG_adni3.nii  (79, 95, 79)    (0.0, 636.0)  (-1418.8359367847443, 26484.93748664856)
